<br/>
<div align="center">
<h2>Bootcamp Data Science — Modulo 2</h2><br/>
<h1>Semana 10 · Deployment de Modelos a Produccion</h1>
<h3>FastAPI + Docker + Azure Container Apps + Ollama local</h3>
<br/>
    <b>Instructor:</b> Jesus Ortiz · SkillNest
</div>
<br/>

Hoy salimos del notebook y pasamos a produccion. Vamos a tomar el modelo de XGBoost que entrenamos para Telco Churn (semana 8) y lo vamos a deployar como una API publica en Azure. Al final de la clase cada uno va a tener su propia URL para mostrar en LinkedIn.

## Objetivos

Al final de la clase van a poder:

1. Entender los conceptos basicos de deployment: endpoint, contenedor, latencia, escalado.
2. Serializar un modelo entrenado de scikit-learn con joblib.
3. Construir una API REST con FastAPI que expone un endpoint /predict.
4. Containerizar la app con Docker.
5. Deployar el contenedor a Azure Container Apps en 6 comandos.
6. Probar el endpoint publico desde Python o desde Swagger UI.
7. Bonus: correr un LLM (Llama 3) en su propia maquina con Ollama.

# 1. Que es deployment?

Hasta ahora, nuestros modelos viven en notebooks. Eso esta bien para experimentar, pero NO sirve para que un negocio use el modelo. Para que el modelo aporte valor real, tiene que estar **disponible** para que otras aplicaciones lo consulten cuando lo necesiten.

**Deployment** significa: tomar el modelo entrenado y exponerlo como un servicio web (API) al que cualquier aplicacion puede mandarle datos y recibir predicciones.

| Estado del modelo | Quien lo usa | Como |
|---|---|---|
| En el notebook | El analista que lo entreno | Manualmente |
| En un script .py | El equipo de datos | Corriendo el script |
| **Deployado como API** | **Cualquier aplicacion** | **HTTP POST** |
| En produccion masiva | Sistema corporativo | Con SLA, monitoreo, autoscaling |

La diferencia entre un Data Analyst y un Data Scientist Senior es que el segundo deja modelos que aportan valor 24/7 sin necesidad de su intervencion.

# 2. Anatomia de un deployment

El stack que vamos a usar:

```
   Usuario (curl, Postman, navegador, otra app)
                 │
                 ▼  HTTPS POST con JSON
   ┌─────────────────────────────────────┐
   │      Azure Container Apps           │   ← Lo que se ve desde internet
   │   (Load balancer, HTTPS, scaling)   │
   └────────────┬────────────────────────┘
                │
                ▼
   ┌─────────────────────────────────────┐
   │      Container Docker               │   ← Tu app + el modelo
   │  ┌────────────────────────────┐     │
   │  │   FastAPI (uvicorn)        │     │
   │  │  ├─ GET  /        (health) │     │
   │  │  ├─ GET  /docs    (swagger)│     │
   │  │  └─ POST /predict          │     │
   │  └────────────────────────────┘     │
   │  ┌────────────────────────────┐     │
   │  │   modelo.joblib (XGBoost)  │     │
   │  └────────────────────────────┘     │
   └─────────────────────────────────────┘
```

Cada pieza:

- **FastAPI**: framework Python para crear APIs. Mucho mas moderno que Flask, mas rapido, con validacion automatica de tipos.
- **uvicorn**: servidor que ejecuta FastAPI en produccion.
- **Docker**: empaqueta toda la app (Python + librerias + modelo) en un contenedor reproducible.
- **Azure Container Apps**: servicio de Azure que ejecuta contenedores con HTTPS, escalado automatico y URL publica, sin que tengamos que gestionar VMs.

# 3. Serializar el modelo con joblib

Antes de deployar tenemos que guardar el modelo entrenado a disco. Para sklearn y XGBoost usamos `joblib`, que es mas eficiente que pickle para objetos grandes.

In [ ]:
# Esto es lo que ya esta hecho en template-telco-api/app/model.joblib
# Aca lo muestro a modo de referencia conceptual:

# import joblib
# from sklearn.pipeline import Pipeline
#
# # Suponiendo que tenemos un pipeline ya entrenado:
# pipeline = Pipeline([
#     ('preprocesamiento', ColumnTransformer([...])),
#     ('modelo', XGBClassifier(...))
# ])
# pipeline.fit(X_train, y_train)
#
# # Serializar a disco
# joblib.dump(pipeline, 'model.joblib')
#
# # Despues, en cualquier proceso (FastAPI, batch job, etc):
# pipeline_cargado = joblib.load('model.joblib')
# prediccion = pipeline_cargado.predict_proba(X_nuevo)

print('IMPORTANTE: serializar el pipeline COMPLETO, no solo el modelo.')
print('Si serializamos solo el modelo, el preprocesamiento tiene que repetirse')
print('manualmente en la API, lo cual es fragil y propenso a errores.')
print()
print('El template del bootcamp ya trae model.joblib con el Pipeline completo')
print('(ColumnTransformer + XGBClassifier) listo para usar.')

# 4. FastAPI: la app web

FastAPI te genera la documentacion automatica (Swagger UI) y valida tipos con Pydantic. Esto es el esqueleto de una API:

```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

# Cargar modelo al iniciar
modelo = joblib.load("model.joblib")

# Definir el formato de input esperado
class Cliente(BaseModel):
    age: int
    tenure: int
    monthly_charges: float

# Crear la app
app = FastAPI()

@app.post("/predict")
def predict(cliente: Cliente):
    df = pd.DataFrame([cliente.model_dump()])
    proba = modelo.predict_proba(df)[0, 1]
    return {"probabilidad_churn": float(proba)}
```

Para correr local:
```bash
uvicorn main:app --host 0.0.0.0 --port 8000
```

Y al hacer GET a http://localhost:8000/docs FastAPI te muestra **Swagger UI**: una interfaz web donde podes probar la API sin escribir codigo.

El template del bootcamp (`template-telco-api/app/main.py`) tiene una version mas completa con:
- Endpoints /predict y /batch
- Validacion exhaustiva de tipos con Literal
- Conversion automatica de strings ("Yes"/"No") al formato que espera el modelo
- Decision de negocio (RIESGO ALTO/MEDIO/BAJO) con recomendacion accionable
- Health check en /

# 5. Docker: empaquetar la app

Docker garantiza que la app funcione igual en cualquier maquina. El `Dockerfile` describe la imagen:

```dockerfile
FROM python:3.10-slim
WORKDIR /code

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app/ ./app/

EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Pasos:
1. **FROM**: imagen base con Python 3.10 (version "slim" pesa solo ~50 MB).
2. **COPY requirements.txt + pip install**: instalar dependencias primero (Docker cachea esto y acelera builds).
3. **COPY app/**: copiar el codigo de la app y el modelo.
4. **EXPOSE 8000**: declarar el puerto donde corre la API.
5. **CMD**: comando que se ejecuta al levantar el contenedor.

Para buildear y correr local:
```bash
docker build -t telco-api .
docker run -p 8000:8000 telco-api
```

Pero **no vamos a buildear local en clase** — Azure ofrece `az acr build` que buildea en el cloud automaticamente. Asi no necesitan tener Docker instalado.

# 6. Deploy a Azure Container Apps

**Azure Container Apps** es el servicio de Azure equivalente a Cloud Run de GCP. Toma una imagen Docker y la deploya con:
- HTTPS automatico
- URL publica
- Escalado automatico (de 0 a N replicas segun trafico)
- Health checks
- Logs y monitoreo

Lo mejor: con `--min-replicas 0` la app **se apaga cuando no hay trafico** y vuelve a arrancar en 5-10 segundos cuando llega un request. Esto reduce el costo a casi cero.

**Los 6 comandos para deployar** (estan en `template-telco-api/deploy.sh`):

```bash
# 1. Resource Group (carpeta logica que contiene todo)
az group create --name rg-telco-NOMBRE --location eastus

# 2. Container Registry (donde vive la imagen)
az acr create --resource-group rg-telco-NOMBRE --name acrnombreXXX --sku Basic --admin-enabled true

# 3. Build la imagen EN AZURE (no necesita Docker local!)
az acr build --registry acrnombreXXX --image telco-api:v1 --file Dockerfile .

# 4. Crear Container Apps Environment
az containerapp env create --name env-telco-NOMBRE --resource-group rg-telco-NOMBRE --location eastus

# 5. Deployar la Container App
az containerapp create \
    --name telco-api-NOMBRE \
    --resource-group rg-telco-NOMBRE \
    --environment env-telco-NOMBRE \
    --image acrnombreXXX.azurecr.io/telco-api:v1 \
    --target-port 8000 \
    --ingress external \
    --min-replicas 0 --max-replicas 2

# 6. Obtener URL publica
az containerapp show \
    --name telco-api-NOMBRE \
    --resource-group rg-telco-NOMBRE \
    --query "properties.configuration.ingress.fqdn" -o tsv
```

El script `deploy.sh` del template los ejecuta todos seguido. Solo tienen que cambiar `TU_NOMBRE` y correr `bash deploy.sh`.

# 7. Probar el endpoint

Cuando termine el deploy van a tener una URL tipo:
```
https://telco-api-bootcamp.eastus.azurecontainerapps.io
```

Tres formas de probar:

### A. Desde el navegador (Swagger UI)
Abrir `https://[URL]/docs` y mandar requests desde la interfaz. FastAPI genera Swagger UI automaticamente.

### B. Desde la terminal (curl)
```bash
curl -X POST https://[URL]/predict \
  -H "Content-Type: application/json" \
  -d '{
    "gender": "Female", "SeniorCitizen": 0,
    "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes",
    "MultipleLines": "No", "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No",
    "StreamingTV": "Yes", "StreamingMovies": "Yes",
    "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.5, "TotalCharges": 195.0
  }'
```

### C. Desde Python (test_api.py)
```bash
python test_api.py https://[URL]
```

El cliente del template (`test_api.py`) prueba 4 escenarios: health, cliente alto riesgo, cliente bajo riesgo, batch.

---
# Ejercicio: deployen su propio modelo

## Reglas
- Trabajo individual.
- Vamos a usar el template `template-telco-api/` como punto de partida.
- Al final cada uno tiene que tener su API funcionando en una URL publica de Azure.

## Pasos

**Parte A — Setup local (15 min)**
1. Clonar/descargar el template `template-telco-api/`.
2. Instalar Azure CLI y hacer `az login`.
3. Correr la app local: `uvicorn app.main:app --reload`. Probar en http://localhost:8000/docs.

**Parte B — Deploy a Azure (30 min)**
4. Editar `deploy.sh` con su nombre en la variable `TU_NOMBRE`.
5. Ejecutar `bash deploy.sh`.
6. Esperar 5-8 minutos.
7. Guardar la URL publica que les devuelve el script.

**Parte C — Probar el endpoint (10 min)**
8. Abrir `https://[URL]/docs` en el navegador.
9. Mandar un request al endpoint `/predict` con el ejemplo del JSON.
10. Correr `python test_api.py https://[URL]` para los 4 escenarios.

**Parte D — Compartir y limpiar (5 min)**
11. Compartir su URL con el companero de al lado para que prueben mutuamente.
12. Al terminar la clase, ejecutar `az group delete --name rg-telco-SUNOMBRE --yes --no-wait` para no acumular costos.

## Bonus (vale puntos extra)

- **Bonus 1**: modificar `app/main.py` para agregar un endpoint `/explicacion` que devuelva las top 3 features que mas influyeron en la prediccion (feature_importances_ del modelo).
- **Bonus 2**: cambiar la decision de RIESGO ALTO/MEDIO/BAJO para que tambien devuelva un descuento sugerido segun el monto (`MonthlyCharges * 0.20` para alto riesgo, etc).
- **Bonus 3**: agregar autenticacion basica con API key en el header (`x-api-key`).
- **Bonus 4**: integrar con Ollama local (ver carpeta `ollama-demo/`) para que la API tambien devuelva un resumen ejecutivo en lenguaje natural.

---
# Bonus: Ollama local (LLM propio sin gastar tokens)

Despues de deployar nuestro modelo de ML clasico, vamos a montar un LLM en su propia maquina para que tengan su propio "ChatGPT" gratis.

**Por que importa esto:**

| | Ollama local | API OpenAI / Anthropic |
|---|---|---|
| Costo | $0 USD/mes despues de setup | $0.50 a $15 USD por 1M tokens |
| Latencia | 50-200ms | 200-1000ms |
| Privacidad | 100% local | Datos pasan por servidores externos |
| Hardware | Necesita 16 GB RAM | Cualquier maquina |

## Lo que YO tengo corriendo: MATEO

Hace un par de meses me arme mi propio asistente LLM local, lo llame **MATEO** (en honor a mi perrito y como acronimo de Modular AI Tooling Engine & Operator). Lo uso a diario para programar, escribir y armar las clases del bootcamp.

**Es 100% open source y lo subi a GitHub para que ustedes lo repliquen.**

Repo: **https://github.com/jeedorsa/MATEO**

Adentro hay:
- Guia paso a paso para montarlo en 30 minutos desde cero: `QUICKSTART-ESTUDIANTES.md`
- Docs detalladas para Mac M3 Pro / Apple Silicon
- Setup opcional en Azure VM con GPU para casos avanzados
- Cliente Python compatible con OpenAI API
- Modelfiles para personalizar el comportamiento (MATEO-coder, MATEO-general)
- Sistema de memoria persistente con RAG
- Scripts para integrar con VS Code, Cursor y otros editores

**Lo que les recomiendo:**
1. Si tienen 16 GB de RAM: empezar con `llama3.1:8b`
2. Si tienen 24+ GB: ir directo a `qwen3:14b`
3. Si tienen 32+ GB: probar `qwen3-coder:30b` para programacion

Despues lo conectan con VS Code via extension Continue y ya tienen su Copilot privado gratis.

## Setup en 3 comandos para arrancar HOY

```bash
# 1. Instalar Ollama
brew install ollama
brew services start ollama

# 2. Descargar un modelo (5-10 min)
ollama pull llama3.1:8b

# 3. Probarlo
ollama run llama3.1:8b "Hola, en una linea: que sabes hacer?"
```

Si funciona ahi, sigan con el [QUICKSTART-ESTUDIANTES.md](https://github.com/jeedorsa/MATEO/blob/main/QUICKSTART-ESTUDIANTES.md) del repo para integrarlo con Python, FastAPI y editores.

## Integracion ML + LLM (la combinacion ganadora)

Pueden combinar AMBAS APIs:
1. Llamar a la **API de Telco Churn** -> recibir probabilidad numerica
2. Pasar esa probabilidad + datos del cliente a **Ollama (MATEO)** -> recibir un resumen ejecutivo en lenguaje natural

Eso es lo que estan haciendo hoy bancos, telcos y retailers: modelo ML clasico para la prediccion + LLM para la explicacion al usuario final.

Ver `ollama-demo/instructions.md` (de esta misma carpeta) para el docker-compose y el codigo del gateway que llama a Ollama.

## Cierre

Hoy aprendimos a:

- Serializar un modelo con joblib (Pipeline completo, no solo el estimador).
- Construir una API REST con FastAPI + validacion de tipos con Pydantic.
- Containerizar con Docker.
- Deployar a Azure Container Apps en 6 comandos.
- Probar el endpoint con Swagger UI, curl y Python.
- Bonus: montar un LLM local con Ollama.

**Lo importante para llevarse:**

Un modelo en un notebook **no aporta valor**. Un modelo deployado como API con URL publica, monitoreo y escalado automatico aporta valor 24/7 sin intervencion humana.

La diferencia entre un Junior Data Scientist y un Senior es saber cerrar el ciclo completo: del notebook a produccion. Esta clase es el cierre de ese ciclo.

**Lo que NO cubrimos hoy** (para que sigan investigando):
- CI/CD para auto-deploy desde GitHub (Azure DevOps, GitHub Actions).
- A/B testing de modelos (deployar 2 versiones, ver cual gana).
- Monitoreo de drift (cuando el modelo deja de funcionar bien con datos nuevos).
- Model Registry para versionar (MLflow, Azure ML).
- Inference batch vs streaming (Kafka, Event Hubs).

Pero con lo de hoy ya pueden mostrar un proyecto productivo en su portfolio. Eso es lo que importa para conseguir trabajo.

Suerte con sus proyectos finales, muchachos. Muahahah.